# 04. 사진 정답 만들기 (천리안 일사량 → 순응도)

**하는 일**: 사진마다 찍은 시각·위경도로 천리안2A 위성 일사량을 받아서, 그 순간의 순응도(A)를 계산해요. 이게 모델③의 정답(y)이에요.

**순서**
1. 사진 목록 읽기 (사진분류 도구 결과가 있으면 입구·출구 사진만)
2. 천리안 요청 (사진마다 1번, 시험 3번 먼저)
3. 정리: 사진 시각 → 일사량 E
4. ASOS로 확인
5. 순응도 계산 → `data/photo_labels.csv`

**준비**: `sunsight_data/.env`의 `KMA_APIHUB_KEY` (04b에서 쓰던 키 그대로). 실행 위치: `sunsight_video` 폴더.

시각·위경도는 여기서 **정답 찾는 데만** 쓰고, 모델 입력에는 넣지 않아요.

In [ ]:
import os, time, math
from pathlib import Path
import numpy as np, pandas as pd, requests

DATA = Path('data')
SD = Path('..') / 'sunsight_data'
CACHE = DATA / 'gk2a_photo'; CACHE.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt'

def env(key):
    p = SD / '.env'
    for line in p.read_text(encoding='utf-8-sig').splitlines():
        if line.strip().startswith(key + '='):
            return line.split('=', 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key)
AUTH_KEY = env('KMA_APIHUB_KEY'); assert AUTH_KEY, 'sunsight_data/.env 에 KMA_APIHUB_KEY 필요'
print('키 확인')

## 1. 사진 목록
- `mapillary_tunnel_candidates.csv`에서 입구 × 촬영 날마다 입구에 가장 가까운 1장 (다운로드한 사진과 같은 규칙)
- `사진분류.csv`(분류 도구 결과)가 있으면 **입구·출구**로 분류한 사진만 남겨요. 없으면 일단 전부

In [ ]:
c = pd.read_csv(DATA / 'mapillary_tunnel_candidates.csv')
c['captured_at'] = pd.to_datetime(c.captured_at.astype(str).str[:19])   # 이미 한국시간(KST)
c['date'] = c.captured_at.dt.date
ph = c.sort_values('dist_m').drop_duplicates(['portal', 'date']).copy()
ph = ph[~ph.tunnel.astype(str).str.contains('지하차도')]          # 지하차도 제외 (형 결정 전 임시)

cls_path = DATA / '사진분류.csv'
if cls_path.exists():
    cls = pd.read_csv(cls_path, dtype={'image_id': str})
    ph['image_id'] = ph.image_id.astype(str)
    ph = ph.merge(cls[['image_id', 'class']], on='image_id', how='inner')
    ph = ph[ph['class'].isin(['입구', '출구'])]
    print('분류 반영:', ph['class'].value_counts().to_dict())
else:
    ph['class'] = '미분류'
    print('사진분류.csv 없음 → 전체 사용 (분류 후 다시 실행)')

ph['t0'] = ph.captured_at.dt.floor('h')
ph['lat_r'], ph['lon_r'] = ph.lat.round(2), ph.lon.round(2)
ph['file'] = [f'{a:.2f}_{b:.2f}_{t:%Y%m%d%H}.txt' for a, b, t in zip(ph.lat_r, ph.lon_r, ph.t0)]
utc = lambda x: (x - pd.Timedelta(hours=9)).strftime('%Y%m%d%H%M')     # API는 UTC
ph['tm1'] = [utc(x - pd.Timedelta(hours=1)) for x in ph.t0]
ph['tm2'] = [utc(x + pd.Timedelta(hours=2)) for x in ph.t0]
print('사진', len(ph), '장 | 연도', ph.captured_at.dt.year.value_counts().sort_index().to_dict())

## 2. 천리안 요청
먼저 **시험 3번**만 출력해요. 값이 숫자로 나오면 다음 셀로. 2019년 이전 사진은 위성 자료가 없을 수 있어요 (빈 값이면 5번에서 ASOS로 대신).

In [ ]:
def fetch(r):
    params = {'int': '30', 'tm1': r.tm1, 'tm2': r.tm2, 'lat': f'{r.lat_r:.4f}', 'lon': f'{r.lon_r:.4f}', 'authKey': AUTH_KEY}
    res = requests.get(BASE_URL, params=params, timeout=60); res.raise_for_status(); return res.text

for r in ph.head(3).itertuples():
    print(r.file, r.captured_at); print(fetch(r)[:800]); print('-' * 60)

In [ ]:
fails = 0
for r in ph.drop_duplicates('file').itertuples():
    f = CACHE / r.file
    if f.exists() and f.stat().st_size > 0: continue
    try: txt = fetch(r)
    except Exception as e: txt = None; print('오류', r.file, e)
    if not txt or 'error' in txt[:200].lower() or '인증' in txt[:200]:
        fails += 1
        if fails >= 5: print('연속 실패 5번 → 멈춤 (한도면 내일 다시)'); break
        continue
    fails = 0; f.write_text(txt, encoding='utf-8'); time.sleep(0.2)
print('받은 파일', sum((CACHE / x).exists() for x in ph.file.unique()), '/', ph.file.nunique())

## 3. 정리: 사진 시각 → 일사량 E
- 응답은 정시마다 1시간 합(MJ/m², 04b에서 ASOS와 맞춰 확인)
- 사진 시각의 **다음 정시** 값 = 그 사진이 찍힌 1시간을 덮는 값 (ASOS·04b와 같은 규칙)

In [ ]:
def parse(txt):
    lines = [l for l in txt.splitlines() if l.strip().startswith('|')]
    head = [x.strip() for x in lines[0].strip('|').split('|')]
    out = []
    for l in lines[1:]:
        d = dict(zip(head, [x.strip() for x in l.strip('|').split('|')]))
        for k in head[4:]:
            out.append({'time_kst': pd.to_datetime(k, format='%Y%m%d%H%M') + pd.Timedelta(hours=9),
                        'gk2a_icsr': pd.to_numeric(d[k], errors='coerce')})
    return out

g = []
for f in ph.file.unique():
    p = CACHE / f
    if p.exists():
        for x in parse(p.read_text(encoding='utf-8')): x['file'] = f; g.append(x)
g = pd.DataFrame(g).dropna().drop_duplicates(['file', 'time_kst'])
ph['tm'] = ph.captured_at.dt.ceil('h')
ph = ph.merge(g, left_on=['file', 'tm'], right_on=['file', 'time_kst'], how='left').drop(columns='time_kst')
print('천리안 값 있음', ph.gk2a_icsr.notna().sum(), '/', len(ph))

## 4. ASOS로 확인
가장 가까운 **일사 재는** 관측소의 같은 시각 값. 2023~2025만 있어요. 둘이 크게 다르면(0.5 MJ 이상) 표시만 하고 지우지는 않아요.

In [ ]:
st = pd.read_csv(SD / 'data' / '1_원본' / 'asos_stations.csv', encoding='utf-8-sig').rename(columns={'지점': 'stn_id', '위도': 'slat', '경도': 'slon'})
aso = pd.read_csv(SD / 'data' / '1_원본' / 'asos_hourly_all.csv', usecols=['stn_id', 'tm', 'icsr', 'dc10Tca'], parse_dates=['tm'])
meas = aso.groupby('stn_id').icsr.max(); st = st[st.stn_id.isin(meas[meas > 0].index)]   # 일사 재는 곳만
def nearest(lat, lon):
    d = np.hypot(st.slat - lat, (st.slon - lon) * math.cos(math.radians(lat)))
    i = d.idxmin(); return st.stn_id[i], d[i] * 111
ph[['stn_id', 'stn_km']] = pd.DataFrame([nearest(a, b) for a, b in zip(ph.lat, ph.lon)], index=ph.index)
ph = ph.merge(aso.rename(columns={'icsr': 'asos_icsr'}), on=['stn_id', 'tm'], how='left')
ok = ph.dropna(subset=['gk2a_icsr', 'asos_icsr'])
print('비교', len(ok), '| 상관', round(ok.gk2a_icsr.corr(ok.asos_icsr), 3))
ph['차이큼'] = (ph.gk2a_icsr - ph.asos_icsr).abs() > 0.5
ph['E'] = ph.gk2a_icsr.fillna(ph.asos_icsr)        # 위성 우선, 없으면 ASOS
ph['E_출처'] = np.where(ph.gk2a_icsr.notna(), '천리안', np.where(ph.asos_icsr.notna(), 'ASOS', '없음'))
print(ph.E_출처.value_counts().to_dict())

## 5. 순응도 계산 (05 노트북과 같은 공식 — 초안, 형 검토 전)
| 단계 | 식 | 근거 |
|---|---|---|
| 바깥 밝기 L20 | E × 277.8 × 126 × 0.15 / π | 일사량 → 조도(126 lm/W) → 시야 휘도(반사율 0.15) |
| 조광 단계 | clip(ceil(L20/4000×4)/4, 0.25, 1) | 국토부 지침 4단계 |
| 입구 밝기 | 280 × 조광 단계 | 지침 (4000 × 0.07) |
| 출구 기준 | 45 | CIE 규칙 |
| A | clip(log10(R)/2, 0, 1) | R=1 → 0, R=100 → 1 |

입구 사진은 A_진입, 출구 사진은 A_진출을 정답으로 써요.

In [ ]:
L20 = ph.E * 277.8 * 126 * 0.15 / np.pi
step = np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)
score = lambda R: np.clip(np.log10(np.where(R > 0, R, 1)) / 2, 0, 1)
ph['L20'] = L20.round(1)
ph['A_진입'] = score(L20 / (280 * step)).round(3)
ph['A_진출'] = score(L20 / 45).round(3)
ph['A'] = np.where(ph['class'] == '출구', ph.A_진출, ph.A_진입)
cols = ['image_id', 'class', 'tunnel', 'side', 'captured_at', 'lat', 'lon', 'heading', 'dist_m',
        'gk2a_icsr', 'asos_icsr', 'stn_km', 'dc10Tca', 'E', 'E_출처', '차이큼', 'L20', 'A_진입', 'A_진출', 'A']
ph[cols].to_csv(DATA / 'photo_labels.csv', index=False, encoding='utf-8-sig')
print('저장: data/photo_labels.csv', len(ph), '장')
ph.groupby('class').A.describe().round(3)